Prueba

In [1]:
pip install hyperopt

Note: you may need to restart the kernel to use updated packages.


In [2]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error, mean_absolute_error
from hyperopt import hp, fmin, tpe, Trials, STATUS_OK
from keras.models import Sequential
from keras.layers import Dense, Dropout, LSTM, InputLayer, Flatten
from keras.callbacks import EarlyStopping, ModelCheckpoint
import matplotlib.pyplot as plt
import tensorflow as tf
from tensorflow.keras.optimizers import Adam
import os
np.random.seed(42)


LEER DATASET

In [3]:
import pandas as pd

datos = pd.read_csv('C:\\Users\\wamt1\\OneDrive\\Escritorio\\colab\\SNpollutionFinal.csv')

datos['date'] = pd.to_datetime(datos['date'])

# Se establece la columna date como index
datos.set_index('date', inplace=True)


In [4]:
datos.head()

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-01 00:00:00,129.0,-21,-11.0,1021.0,2,1.79
2010-01-01 01:00:00,129.0,-21,-12.0,1020.0,2,4.92
2010-01-01 02:00:00,129.0,-21,-11.0,1019.0,2,6.71
2010-01-01 03:00:00,129.0,-21,-14.0,1019.0,2,9.84
2010-01-01 04:00:00,129.0,-20,-12.0,1018.0,2,12.97


Se eliminan las primeras 24 filas debido a que estas contenían valores NAN en la columna pollution, y habían sido rellenadas con interpolación lineal.

In [5]:
datos = datos.drop(datos.index[:24])

datos

,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,129.0,-16,-4.0,1020.0,1,1.79
2010-01-02 01:00:00,148.0,-15,-4.0,1020.0,1,2.68
2010-01-02 02:00:00,159.0,-11,-5.0,1021.0,1,3.57
2010-01-02 03:00:00,181.0,-7,-5.0,1022.0,1,5.36
2010-01-02 04:00:00,138.0,-7,-5.0,1022.0,1,6.25
...,...,...,...,...,...,...
2014-12-31 19:00:00,8.0,-23,-2.0,1034.0,2,231.97
2014-12-31 20:00:00,10.0,-22,-3.0,1034.0,2,237.78
2014-12-31 21:00:00,10.0,-22,-3.0,1034.0,2,242.70


In [6]:
#Se verifica que el index está en formato de dato "datetime64"
print("El index del dataframe input es un tipo de dato: ", datos.index.dtype)


El index del dataframe input es un tipo de dato:  datetime64[ns]


Se divide el dataset

In [7]:
# Dividir el conjunto de datos en entrenamiento y prueba
train, test = train_test_split(datos, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
test, val = train_test_split(test, test_size=0.33, shuffle=False)

print("Las dimensiones de train son: ", train.shape)
print("Las dimensiones de test son: ", test.shape)
print("Las dimensiones de val son: ", val.shape)

Las dimensiones de train son:  (30660, 6)
Las dimensiones de test son:  (8803, 6)
Las dimensiones de val son:  (4337, 6)


Se normalizan los datos

In [8]:
from sklearn.preprocessing import StandardScaler


# Normalizar solo con los datos de entrenamiento
scaler = StandardScaler()
train = scaler.fit_transform(train)

# Aplicar la transformación a test y val usando los parámetros de train
test = scaler.transform(test)
val = scaler.transform(val)

train = pd.DataFrame(train, columns=datos.columns)
test = pd.DataFrame(test, columns=datos.columns)
val = pd.DataFrame(val, columns=datos.columns)


Se unen los datos nuevamente, ahora normalizados, en un único conjunto.

In [9]:
datosNormalizados = pd.concat([train, test, val])

datosNormalizados.index = datos.index


In [10]:
datosNormalizados.shape

(43800, 6)

In [11]:
datosNormalizados.head(10)


,pollution,dew,temp,press,wnd_dir,wnd_spd
date,,,,,,
2010-01-02 00:00:00,0.317681,-1.214023,-1.268524,0.329687,-0.380944,-0.464048
2010-01-02 01:00:00,0.526152,-1.144302,-1.268524,0.329687,-0.380944,-0.446575
2010-01-02 02:00:00,0.646846,-0.865419,-1.349314,0.426127,-0.380944,-0.429103
2010-01-02 03:00:00,0.888234,-0.586536,-1.349314,0.522567,-0.380944,-0.393962
2010-01-02 04:00:00,0.416431,-0.586536,-1.349314,0.522567,-0.380944,-0.376489
2010-01-02 05:00:00,0.098238,-0.586536,-1.430104,0.522567,-0.380944,-0.359017
2010-01-02 06:00:00,0.054349,-0.586536,-1.430104,0.619008,-0.380944,-0.323876
2010-01-02 07:00:00,0.262820,-0.586536,-1.349314,0.715448,-0.380944,-0.288735
2010-01-02 08:00:00,0.218931,-0.656257,-1.430104,0.715448,-0.380944,-0.253594


Espacio de búsqueda

In [12]:
space = {
    'layers': hp.quniform('layers', 1, 4, 1), # Cantidad de capas LSTM
    'units': hp.choice('units', [2 ** i for i in range(3, 8)]),  # Número de unidades LSTM
    'activation': hp.choice('activation', ['tanh', 'sigmoid', 'relu', 'linear']),
    'dropout': hp.quniform('dropout', 0, 0.5, 0.1),  # Dropout para regularización
    'learning_rate': hp.loguniform('learning_rate', np.log(0.000001), np.log(0.01)),  # Tasa de aprendizaje
    'epochs': hp.choice('epochs', [2 ** i for i in range(3, 9)]),  # Número de épocas de entrenamiento
    'batch': hp.choice('batch',[2 ** i for i in range(3, 9)])
}

Se establece el formato de datos de entrada para redes LSTM, es decir [observaciones, retardos, caracteristicas]

In [13]:
futuros = 24
pasados  = 12

In [14]:
datosX = []
datosY = []
for i in range(pasados, len(datosNormalizados) - futuros + 1):
  datosX.append(datosNormalizados.iloc[i-pasados:i, 0:datosNormalizados.shape[1]])
  datosY.append(datosNormalizados.iloc[i+futuros-1:i+futuros, 0])


In [15]:
# Convertir las listas en arrays numpy
datosX = np.array(datosX)
datosY = np.array(datosY)

# Ver las dimensiones (shape) de los arrays
print("Dimensiones de X:", datosX.shape)  # (n_muestras, pasos_de_tiempo, n_características)
print("Dimensiones de Y:", datosY.shape)  # (n_muestras, n_características)

Dimensiones de X: (43765, 12, 6)
Dimensiones de Y: (43765, 1)


In [16]:
print(datosX[0])

[[ 0.31768099 -1.2140229  -1.26852411  0.32968671 -0.38094383 -0.46404777]
 [ 0.52615226 -1.14430217 -1.26852411  0.32968671 -0.38094383 -0.44657536]
 [ 0.64684616 -0.86541928 -1.34931411  0.42612698 -0.38094383 -0.42910295]
 [ 0.88823396 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.39396181]
 [ 0.41643054 -0.58653639 -1.34931411  0.52256725 -0.38094383 -0.3764894 ]
 [ 0.09823754 -0.58653639 -1.4301041   0.52256725 -0.38094383 -0.35901698]
 [ 0.05434885 -0.58653639 -1.4301041   0.61900753 -0.38094383 -0.32387584]
 [ 0.26282012 -0.58653639 -1.34931411  0.7154478  -0.38094383 -0.2887347 ]
 [ 0.21893143 -0.65625711 -1.4301041   0.7154478  -0.38094383 -0.25359356]
 [ 0.3505975  -0.58653639 -1.34931411  0.81188808 -0.38094383 -0.21845241]
 [ 0.43837488 -0.58653639 -1.34931411  0.90832835 -0.38094383 -0.15700449]
 [ 0.57004095 -0.65625711 -1.34931411  0.90832835 -0.38094383 -0.09555657]]


Se dividen nuevamente los conjuntos de datos

In [17]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainX, testX = train_test_split(datosX, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testX, valX = train_test_split(testX, test_size=0.33, shuffle=False)

print("Las dimensiones de trainX son: ", trainX.shape)
print("Las dimensiones de testX son: ", testX.shape)
print("Las dimensiones de valX son: ", valX.shape)


Las dimensiones de trainX son:  (30635, 12, 6)
Las dimensiones de testX son:  (8797, 12, 6)
Las dimensiones de valX son:  (4333, 12, 6)


In [18]:
from sklearn.model_selection import train_test_split

# Dividir el conjunto de datos en entrenamiento y prueba
trainY, testY = train_test_split(datosY, test_size=0.3, shuffle=False)

# Luego, dividir el conjunto de prueba en conjuntos de prueba y validación
testY, valY = train_test_split(testY, test_size=0.33, shuffle=False)

print("Las dimensiones de trainY son: ", trainY.shape)
print("Las dimensiones de testY son: ", testY.shape)
print("Las dimensiones de valY son: ", valY.shape)

Las dimensiones de trainY son:  (30635, 1)
Las dimensiones de testY son:  (8797, 1)
Las dimensiones de valY son:  (4333, 1)


Se crean métricas para medir desempeño

In [19]:
import tensorflow.keras.backend as K

def smape(y_true, y_pred):
    """
    Define la función SMAPE (Error Porcentual Absoluto Medio Simétrico).
    """
    summ = K.abs(y_true) + K.abs(y_pred)
    smape_val = K.abs(y_pred - y_true) / summ * 2.0
    return K.mean(smape_val, axis=-1)

def rmse(y_true, y_pred):
    return K.sqrt(K.mean(K.square(y_pred - y_true)))

def ia(y_true, y_pred):
    numerator = K.sum(K.abs(y_true - y_pred))
    denominator = K.sum(K.abs(y_true - K.mean(y_true)) + K.abs(y_pred - K.mean(y_true)))
    return 1 - (numerator / denominator)

In [20]:
from sklearn.metrics import mean_squared_error as mse
from sklearn.metrics import mean_absolute_error as mae

def graficarPrediccion(modelo, x, y, inicio, final):
  predicciones = modelo.predict(x)
  predicciones = predicciones.flatten()
  df = pd.DataFrame({'Originales': y, 'Predichos': predicciones})
  plt.plot(df.index, df['Originales'][inicio:final], label='Originales')
  plt.plot(df.index, df['Predichos'][inicio:final], label='Predichos')
  return df, mse(y, predicciones),  mae(y, predicciones), rmse(y, predicciones), smape(y,predicciones), ia(y, predicciones)

Versión Final


In [21]:
def objective(params):

    model = Sequential()
    model.add(InputLayer(input_shape=(trainX.shape[1], trainX.shape[2])))
    if (params['layers'] == 1):
      model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=False))
      model.add(Dropout(params['dropout']))

    else:
      for _ in range(int(params['layers']) - 1):
          model.add(LSTM(units=params['units'], activation=params['activation'], return_sequences=True))
          model.add(Dropout(params['dropout']))
      model.add(LSTM(units=params['units'], activation=params['activation']))
      model.add(Dropout(params['dropout']))

    model.add(Dense(1))


    opt = Adam(learning_rate=params['learning_rate'])
    model.compile(optimizer=opt, loss='mse', metrics=["mae", smape, rmse, ia])

    early_stopping = EarlyStopping(monitor='val_loss', patience=15, verbose=1, restore_best_weights= True)

    model.fit(trainX, trainY, epochs=params['epochs'],
                        validation_data=(testX, testY),
                        verbose = 2, batch_size=params['batch'], callbacks=[early_stopping])


    predictions = model.predict(valX)


    loss = mean_squared_error(valY, predictions)

    return {'loss': loss, 'status': STATUS_OK}

In [22]:
trials = Trials()
best = fmin(objective, space, algo=tpe.suggest, max_evals=50, trials=trials, rstate=np.random.default_rng(42))

  0%|          | 0/50 [00:00<?, ?trial/s, best loss=?]

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/128                                           

958/958 - 43s - 45ms/step - ia: 0.2399 - loss: 0.9354 - mae: 0.7228 - rmse: 0.9470 - smape: 1.5648 - val_ia: 0.2466 - val_loss: 0.8686 - val_mae: 0.6920 - val_rmse: 0.7803 - val_smape: 1.4890

Epoch 2/128                                           

958/958 - 16s - 16ms/step - ia: 0.2814 - loss: 0.9056 - mae: 0.7088 - rmse: 0.9310 - smape: 1.4769 - val_ia: 0.2497 - val_loss: 0.8459 - val_mae: 0.6823 - val_rmse: 0.7728 - val_smape: 1.4285

Epoch 3/128                                           

958/958 - 14s - 15ms/step - ia: 0.2964 - loss: 0.8922 - mae: 0.7020 - rmse: 0.9255 - smape: 1.4487 - val_ia: 0.2493 - val_loss: 0.8401 - val_mae: 0.6844 - val_rmse: 0.7756 - val_smape: 1.4308

Epoch 4/128                                           

958/958 - 15s - 15ms/step - ia: 0.3142 - loss: 0.8765 - mae: 0.6958 - rmse: 0.9178 - smape: 1.4226 - val_ia: 0.2466 - val_loss: 0.8345 - val_mae: 0.6871 - val_rmse: 0.7800 - val_smape: 1.4130

Epoc

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                          

120/120 - 59s - 495ms/step - ia: 0.2712 - loss: 0.8907 - mae: 0.6986 - rmse: 0.9406 - smape: 1.4669 - val_ia: 0.2969 - val_loss: 0.8057 - val_mae: 0.6710 - val_rmse: 0.8255 - val_smape: 1.3478

Epoch 2/16                                                                          

120/120 - 34s - 282ms/step - ia: 0.3737 - loss: 0.8205 - mae: 0.6677 - rmse: 0.9037 - smape: 1.2972 - val_ia: 0.3185 - val_loss: 0.8048 - val_mae: 0.6785 - val_rmse: 0.8336 - val_smape: 1.3497

Epoch 3/16                                                                          

120/120 - 27s - 223ms/step - ia: 0.4050 - loss: 0.7900 - mae: 0.6513 - rmse: 0.8867 - smape: 1.2530 - val_ia: 0.3452 - val_loss: 0.7981 - val_mae: 0.6725 - val_rmse: 0.8340 - val_smape: 1.3086

Epoch 4/16                                                                          

120/120 - 43s - 356ms/step - ia: 0.4171 - loss: 0.7751 - mae: 0.6445 - rms

...

Epoch 1/8                                                                            

3830/3830 - 35s - 9ms/step - ia: 0.3448 - loss: 0.8474 - mae: 0.6813 - rmse: 0.8609 - smape: 1.3771 - val_ia: 0.1991 - val_loss: 0.8047 - val_mae: 0.6648 - val_rmse: 0.7049 - val_smape: 1.3159

Epoch 2/8                                                                            

3830/3830 - 30s - 8ms/step - ia: 0.3833 - loss: 0.8110 - mae: 0.6610 - rmse: 0.8420 - smape: 1.2946 - val_ia: 0.1996 - val_loss: 0.7741 - val_mae: 0.6534 - val_rmse: 0.6925 - val_smape: 1.2939

Epoch 3/8                                                                            

3830/3830 - 43s - 11ms/step - ia: 0.3949 - loss: 0.7928 - mae: 0.6523 - rmse: 0.8344 - smape: 1.2661 - val_ia: 0.2014 - val_loss: 0.7847 - val_mae: 0.6571 - val_rmse: 0.6971 - val_smape: 1.2908

Epoch 4/8                                                                            

3830/3830 - 31s - 8ms/step - ia: 0.4000 - loss: 0.7830 - mae: 0.6481 

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/16                                                                           

240/240 - 10s - 41ms/step - ia: 0.3493 - loss: 0.8910 - mae: 0.6972 - rmse: 0.9360 - smape: 1.3492 - val_ia: 0.3102 - val_loss: 0.7599 - val_mae: 0.6475 - val_rmse: 0.7897 - val_smape: 1.2946

Epoch 2/16                                                                           

240/240 - 2s - 8ms/step - ia: 0.3844 - loss: 0.8176 - mae: 0.6662 - rmse: 0.8998 - smape: 1.2938 - val_ia: 0.3341 - val_loss: 0.7665 - val_mae: 0.6484 - val_rmse: 0.7997 - val_smape: 1.2794

Epoch 3/16                                                                           

240/240 - 3s - 11ms/step - ia: 0.3964 - loss: 0.8036 - mae: 0.6565 - rmse: 0.8919 - smape: 1.2763 - val_ia: 0.3302 - val_loss: 0.7602 - val_mae: 0.6395 - val_rmse: 0.7888 - val_smape: 1.2560

Epoch 4/16                                                                           

240/240 - 3s - 11ms/step - ia: 0.4018 - loss: 0.7909 - mae: 0.6511 - rmse: 0

c:\Users\wamt1\AppData\Local\Programs\Python\Python312\Lib\site-packages\keras\src\layers\core\input_layer.py:26: UserWarning: Argument `input_shape` is deprecated. Use `shape` instead.
  warnings.warn(



Epoch 1/32                                                                           

120/120 - 12s - 100ms/step - ia: 0.2688 - loss: 0.9107 - mae: 0.7080 - rmse: 0.9506 - smape: 1.4775 - val_ia: 0.2987 - val_loss: 0.7927 - val_mae: 0.6622 - val_rmse: 0.8186 - val_smape: 1.3858

Epoch 2/32                                                                           

120/120 - 1s - 12ms/step - ia: 0.3237 - loss: 0.8610 - mae: 0.6855 - rmse: 0.9255 - smape: 1.3901 - val_ia: 0.3110 - val_loss: 0.7715 - val_mae: 0.6591 - val_rmse: 0.8135 - val_smape: 1.3452

Epoch 3/32                                                                           

120/120 - 2s - 14ms/step - ia: 0.3429 - loss: 0.8449 - mae: 0.6776 - rmse: 0.9174 - smape: 1.3586 - val_ia: 0.3237 - val_loss: 0.7650 - val_mae: 0.6494 - val_rmse: 0.8065 - val_smape: 1.3191

Epoch 4/32                                                                           

120/120 - 2s - 13ms/step - ia: 0.3511 - loss: 0.8430 - mae: 0.6760 - rmse:

In [23]:
print(best)

{'activation': 1, 'batch': 2, 'dropout': 0.30000000000000004, 'epochs': 5, 'layers': 4.0, 'learning_rate': 0.0002283375510850933, 'units': 4}
